In [ ]:
import warnings
import platform

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from IPython.display import display

warnings.filterwarnings("ignore")

if platform.system() == "Windows":
    plt.rc("font", family="Malgun Gothic")
else:
    font_path = "/usr/share/fonts/truetype/nanum/NanumBarunGothicBold.ttf"
    font_name = fm.FontProperties(fname=font_path).get_name()
    plt.rc("font", family=font_name)

plt.rcParams["axes.unicode_minus"] = False
print(f"현재 설정된 폰트: {plt.rcParams['font.family']}")
            


In [ ]:
DATA_PATH = r"../data/processed/new_process.csv"
df_raw = pd.read_csv(DATA_PATH)
df_raw.head()
            


In [ ]:
plt.figure(figsize=(14, 6))
sns.heatmap(df_raw.isnull(), cbar=False)
plt.title("결측치 패턴 Heatmap")
plt.show()

null_summary = df_raw.isnull().sum().sort_values(ascending=False)
display(null_summary[null_summary > 0])

df_null = df_raw[df_raw.isnull().any(axis=1)]
display(df_null.head())
print(f"결측치가 포함된 행 수: {len(df_null):,}")
            


In [ ]:
groups = {
    "OCV": [
        "ocv1_ocv", "ocv2_ocv", "socv1_ocv", "socv2_ocv", "socv3_ocv",
        "ocv1_deltaocv", "ocv2_deltaocv",
    ],
    "CHARGE": [
        "c1_curr_end", "c1_voltage_avg", "c1_capa", "c1_ccval", "c1_time_cc",
        "c2_curr_end", "c2_voltage_avg", "c2_capa", "c2_ccval", "c2_time_cc",
        "c3_curr_end", "c3_voltage_avg", "c3_capa", "c3_ccval", "c3_time_cc",
        "c4_curr_end", "c4_voltage_avg", "c4_capa", "c4_ccval", "c4_time_cc",
    ],
    "DISCHARGE": [
        "dc1_curr_end", "dc1_voltage_avg", "dc1_capa", "dc1_capafit",
        "dc2_curr_end", "dc2_voltage_avg", "dc2_capa", "dc2_capafit",
        "dc3_curr_end", "dc3_voltage_avg", "dc3_capa", "dc3_capafit",
    ],
    "TEMP": [
        "c1_temp_avg", "dc1_temp_avg",
        "c2_temp_avg", "dc2_temp_avg",
        "c3_temp_avg", "dc3_temp_avg",
        "c4_temp_avg", "pg1_temp_avg",
    ],
    "IMPEDANCE": ["pg1_impfit", "pg1_imp", "pc1_imp", "m1_res_ac"],
}

tat_groups = {
    "RTA": ["rta1_tat", "rta2_tat"],
    "HTA": ["hta1_tat"],
    "OCV": ["ocv1_tat", "ocv2_tat"],
    "CHARGE": ["c1_tat", "c2_tat", "c3_tat", "c4_tat"],
    "DISCHARGE": ["dc1_tat", "dc2_tat", "dc3_tat"],
    "POST": ["pg1_tat", "pc1_tat"],
    "AGING": ["sa1_tat", "sa2_tat", "sa3_tat"],
    "SOCV": ["socv1_tat", "socv2_tat", "socv3_tat"],
}
            


In [ ]:
def plot_group_panels(df, group_map, plot_kind, title_suffix, cols_per_row=4, sample_size=None):
    for group_name, cols in group_map.items():
        valid_cols = [col for col in cols if col in df.columns]
        if len(valid_cols) < 1:
            continue

        plot_df = df[valid_cols]
        if sample_size is not None and len(plot_df) > sample_size:
            plot_df = plot_df.sample(n=sample_size, random_state=42)

        n_cols = min(cols_per_row, len(valid_cols))
        n_rows = int(np.ceil(len(valid_cols) / n_cols))
        fig, axes = plt.subplots(n_rows, n_cols, figsize=(4 * n_cols, 4 * n_rows))
        axes = np.array(axes).reshape(-1)
        fig.suptitle(f"{group_name} {title_suffix}", fontsize=16)

        for ax, col in zip(axes, valid_cols):
            if plot_kind == "box":
                sns.boxplot(x=plot_df[col], ax=ax)
            elif plot_kind == "hist":
                sns.histplot(plot_df[col], kde=True, ax=ax)
            else:
                raise ValueError(f"지원하지 않는 plot_kind: {plot_kind}")
            ax.set_title(col)

        for ax in axes[len(valid_cols):]:
            ax.remove()

        plt.tight_layout(rect=[0, 0, 1, 0.96])
        plt.show()
            


In [ ]:
plot_group_panels(df_raw, groups, plot_kind="box", title_suffix="변수 이상치 분석")
plot_group_panels(df_raw, groups, plot_kind="hist", title_suffix="분포 분석")
            


In [ ]:
for group_name, cols in groups.items():
    valid_cols = [col for col in cols if col in df_raw.columns]
    if len(valid_cols) < 2:
        continue

    sample_df = df_raw[valid_cols].dropna().sample(
        n=min(2000, len(df_raw[valid_cols].dropna())),
        random_state=42,
    )
    if sample_df.empty:
        continue

    sns.pairplot(sample_df)
    plt.suptitle(f"{group_name} 변수 관계 분석 (Pairplot)", y=1.02)
    plt.show()

key_pairs = [
    ("c1_capa", "dc1_capa"),
    ("c2_capa", "dc2_capa"),
    ("dc1_voltage_avg", "dc1_capa"),
    ("dc2_voltage_avg", "dc2_capa"),
    ("dc3_voltage_avg", "dc3_capa"),
]

plt.figure(figsize=(15, 10))
for i, (x_col, y_col) in enumerate(key_pairs, start=1):
    if x_col in df_raw.columns and y_col in df_raw.columns:
        plt.subplot(3, 2, i)
        sns.scatterplot(data=df_raw, x=x_col, y=y_col, alpha=0.3)
        plt.title(f"{x_col} vs {y_col}")
plt.tight_layout()
plt.show()

for group_name, cols in groups.items():
    valid_cols = [col for col in cols if col in df_raw.columns]
    if len(valid_cols) < 2:
        continue

    corr = df_raw[valid_cols].corr()
    plt.figure(figsize=(8, 6))
    sns.heatmap(corr, annot=False, cmap="coolwarm")
    plt.title(f"{group_name} 상관관계 Heatmap")
    plt.show()
            


In [ ]:
def detect_outliers_refined(df, cols, iqr_multiplier=3.0, min_variance=1e-6, min_ratio=1.0):
    results = []

    for col in cols:
        if col not in df.columns:
            continue

        series = df[col].dropna()
        if series.empty or not pd.api.types.is_numeric_dtype(series):
            continue
        if series.var() < min_variance:
            continue

        q1 = series.quantile(0.25)
        q3 = series.quantile(0.75)
        iqr = q3 - q1
        if iqr == 0:
            continue

        lower = q1 - iqr_multiplier * iqr
        upper = q3 + iqr_multiplier * iqr
        outlier_mask = (series < lower) | (series > upper)
        outlier_count = int(outlier_mask.sum())
        outlier_ratio = (outlier_count / len(series)) * 100

        if outlier_ratio >= min_ratio:
            results.append(
                {
                    "column": col,
                    "outlier_count": outlier_count,
                    "outlier_ratio(%)": round(outlier_ratio, 2),
                }
            )

    if not results:
        return pd.DataFrame(columns=["column", "outlier_count", "outlier_ratio(%)"])

    return pd.DataFrame(results).sort_values(by="outlier_ratio(%)", ascending=False)
            


In [ ]:
cols_raw = sum(groups.values(), [])
refined_outliers = detect_outliers_refined(
    df_raw,
    cols_raw,
    iqr_multiplier=3.0,
    min_variance=1e-5,
    min_ratio=1.0,
)

print("=== 전기적 특성 이상치 ===")
display(refined_outliers)

cols_tat = sum(tat_groups.values(), [])
refined_outliers_tat = detect_outliers_refined(
    df_raw,
    cols_tat,
    iqr_multiplier=3.0,
    min_variance=1e-5,
    min_ratio=1.0,
)

print("=== 공정 시간(TAT) 이상치 ===")
display(refined_outliers_tat)
            


In [ ]:
def build_outlier_report(df_outliers, data_name, var_type):
    report = df_outliers.copy()
    if report.empty:
        return pd.DataFrame(
            columns=[
                "데이터명", "column", "의미", "유형", "outlier_count",
                "outlier_ratio(%)", "추정원인", "정제방안",
            ]
        )

    report["데이터명"] = data_name
    report["유형"] = var_type

    def classify_meaning(col):
        if "voltage" in col:
            return "전압"
        if "capa" in col:
            return "용량"
        if "curr" in col:
            return "전류"
        if "temp" in col:
            return "온도"
        if "tat" in col:
            return "공정시간"
        if "ocv" in col:
            return "개방전압"
        return "기타"

    report["의미"] = report["column"].apply(classify_meaning)
    report["추정원인"] = ""
    report["정제방안"] = ""

    return report[
        [
            "데이터명",
            "column",
            "의미",
            "유형",
            "outlier_count",
            "outlier_ratio(%)",
            "추정원인",
            "정제방안",
        ]
    ]


report_raw = build_outlier_report(refined_outliers, data_name="df_raw", var_type="전기적 특성")
report_tat = build_outlier_report(refined_outliers_tat, data_name="df_raw", var_type="공정 시간(TAT)")
final_report = pd.concat([report_raw, report_tat], ignore_index=True)
display(final_report)
            


In [ ]:
analysis_df = pd.read_csv(r"../data/processed/bat_process.csv", encoding="euc-kr")
analysis_df["judge_bin"] = analysis_df["judge"].astype(str).str.contains("불량").astype(int)

A_DATA_ERROR = [
    "ocv1_ocv", "ocv2_ocv", "socv1_ocv", "socv2_ocv", "socv3_ocv",
    "c1_time_cc", "c2_time_cc", "c3_time_cc", "c4_time_cc",
    "c3_time_cv", "c4_time_cv", "pg1_imp", "pg1_impfit", "pc1_imp",
    "m1_res_ac", "m1_thick", "c1_temp_avg", "dc1_temp_avg",
    "c2_temp_avg", "dc2_temp_avg", "c3_temp_avg", "dc3_temp_avg", "c4_temp_avg",
]
B_RECIPE = [
    "c1_curr_end", "dc1_curr_end", "c2_curr_end", "dc2_curr_end",
    "c3_curr_end", "dc3_curr_end", "c4_curr_end", "c3_cvval", "c4_cvval", "c3_ccval", "c4_ccval",
]
C_PROCESS = ["ocv2_deltaocv", "pg1_imp", "pc1_imp", "m1_res_ac", "m1_thick", "c3_time_cv", "c4_time_cv"]
D_NORMAL = [
    "c1_temp_avg", "dc1_temp_avg", "c2_temp_avg", "dc2_temp_avg", "c3_temp_avg", "dc3_temp_avg", "c4_temp_avg",
    "c1_time_cc", "c2_time_cc", "c3_time_cc", "c4_time_cc", "c3_time_cv", "c4_time_cv",
    "c1_voltage_avg", "dc1_voltage_avg", "c2_voltage_avg", "dc2_voltage_avg", "c3_voltage_avg", "dc3_voltage_avg", "c4_voltage_avg",
]

def get_existing(cols):
    return [col for col in cols if col in analysis_df.columns]

A_DATA_ERROR = get_existing(A_DATA_ERROR)
B_RECIPE = get_existing(B_RECIPE)
C_PROCESS = get_existing(C_PROCESS)
D_NORMAL = get_existing(D_NORMAL)

summary_rows = []

for col in A_DATA_ERROR:
    series = analysis_df[col].dropna()
    if series.empty:
        continue
    if "temp" in col:
        low, high = 0, 1000
    elif "time" in col:
        low, high = 0, 100000
    elif "ocv" in col or "voltage" in col:
        low, high = 0, 5000
    elif "imp" in col or "res" in col or "thick" in col:
        low, high = 0, 10000
    else:
        low, high = None, None
    invalid = ((analysis_df[col] < low) | (analysis_df[col] > high)).sum() if low is not None else 0
    summary_rows.append({"column": col, "type": "A_data_error", "changed_count": int(invalid)})

for col in B_RECIPE:
    rounded = analysis_df[col].dropna().round(0)
    if rounded.empty:
        continue
    value_ratio = rounded.value_counts(normalize=True)
    summary_rows.append({"column": col, "type": "B_recipe", "changed_count": int((value_ratio < 0.03).sum())})

for col in C_PROCESS + D_NORMAL:
    series = analysis_df[col].dropna()
    if len(series) < 10:
        continue
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    if iqr == 0:
        continue
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    changed = ((analysis_df[col] < lower) | (analysis_df[col] > upper)).sum()
    summary_rows.append(
        {
            "column": col,
            "type": "C_process" if col in C_PROCESS else "D_normal",
            "changed_count": int(changed),
            "lower": round(lower, 3),
            "upper": round(upper, 3),
        }
    )

summary_df = pd.DataFrame(summary_rows).sort_values(["type", "changed_count"], ascending=[True, False])
display(summary_df.head(50))

type_counts = summary_df.groupby("type")["changed_count"].sum().reset_index()
plt.figure(figsize=(8, 4))
plt.bar(type_counts["type"], type_counts["changed_count"], color=["#E45756", "#4C78A8", "#72B7B2", "#54A24B"])
plt.title("이상치/정제 대상 건수 요약")
plt.ylabel("건수")
plt.tight_layout()
plt.show()

top_changed = summary_df.sort_values("changed_count", ascending=False).head(12)
plt.figure(figsize=(10, 6))
plt.barh(top_changed["column"], top_changed["changed_count"], color="#F58518")
plt.gca().invert_yaxis()
plt.title("정제 대상 상위 변수")
plt.xlabel("건수")
plt.tight_layout()
plt.show()
        
